# 02 - Build the index

**Questions this notebook answers:** how are documents turned into chunks, how are chunks embedded, how is the FAISS index built, and how do we know it is correct?

```
01 Data + KB  ->  02 Build index  ->  03 RAG pipeline  ->  04 Evaluation
                    (you are here)
```

```
documents --split--> chunks --embed--> vectors --index--> FAISS
```

The code is in `src/kb.py`. This notebook runs it step by step and checks the result.

## Decisions

| Step | Choice | Why |
|---|---|---|
| Chunking | Pack whole sentences into chunks of at most **1000 characters**; repeat the last sentence (if it is at most 200 characters) at the start of the next chunk | About 250 tokens, well under the encoder's 512-token limit, and usually one idea per chunk. Splitting on sentences avoids cutting a fact in half. |
| Question on every chunk | The document's question (`What is Type 2 Diabetes?`) is put on top of every chunk | A chunk from the middle of a long answer would otherwise not say what it is about. It is added **after** splitting, so a chunk can never be only a question. This is a design guess; notebook 04 (section 4) measures it against chunks without the question. |
| Embedding model | `BAAI/bge-m3` | A strong general-purpose encoder. No other encoder was compared; to try one, change `EMBEDDING_MODEL` in `src/config.py`, rebuild here and rerun notebook 04. |
| Normalised vectors | Yes | With unit-length vectors, inner product equals cosine similarity. |
| Index | `IndexFlatIP`, exact search | Fast enough at this size, and there is no approximation to explain or tune. |

Chunk size and overlap are reasonable defaults, not tuned values.

In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import config
from src.kb import MIN_CHUNK_CHARS, build_chunks, build_index, encode_texts, load_encoder, save_index

kb = pd.read_csv(config.KB_PATH, keep_default_na=False)
print(f"{len(kb):,} documents from {config.KB_PATH.relative_to(ROOT)}")

## 1. Chunk

In [ ]:
chunks = build_chunks(kb)

per_document = chunks.groupby("doc_id").size()
print(f"{len(kb):,} documents -> {len(chunks):,} chunks")
print(f"documents with no chunk: {len(kb) - len(per_document):,}")
print(f"chunks per document: median {per_document.median():.0f}, max {per_document.max()}")

for _, row in chunks.sample(2, random_state=0).iterrows():
    print(f"\n[chunk {row['chunk_id']}] {row['source']} | {row['title']} | {row['url']}")
    print(row["text"][:500])

### Checks before spending time on embeddings

* No chunk is only a header.
* Chunk sizes look as intended.
* Chunks fit the encoder's token window, so nothing is silently cut off.

In [ ]:
bodies = chunks["text"].str.split("\n", n=1).str[1].fillna("")
assert (bodies.str.len() >= MIN_CHUNK_CHARS).all(), "a chunk has (almost) no content"

lengths = chunks["text"].str.len()
print(f"characters per chunk: median {lengths.median():.0f}, 95th percentile {lengths.quantile(.95):.0f}, max {lengths.max()}")
lengths.plot.hist(bins=50, figsize=(7, 3), title="Characters per chunk")
plt.tight_layout()
plt.show()

encoder = load_encoder()
sample = chunks["text"].sample(min(2000, len(chunks)), random_state=0).tolist()
token_counts = np.array([len(ids) for ids in encoder.tokenizer(sample)["input_ids"]])
print(f"tokens per chunk (2000 sampled): max {token_counts.max()}, share over {encoder.max_seq_length}: {(token_counts > encoder.max_seq_length).mean():.2%}")

## 2. Embed and index

Use a GPU if you have one. bge-m3 is a large model, so encoding on a CPU is slow.

In [ ]:
embeddings = encode_texts(encoder, chunks["text"])
print("embeddings:", embeddings.shape, embeddings.dtype)
print("mean vector length (should be about 1.0):", float(np.linalg.norm(embeddings, axis=1).mean()))

index = build_index(embeddings)
save_index(index, chunks)   # refuses to save if the vectors and the chunk table do not line up
print(f"saved {index.ntotal:,} vectors")
print(f"index file {config.INDEX_PATH.stat().st_size / 1024**2:.0f} MB, chunk table {config.CHUNKS_PATH.stat().st_size / 1024**2:.0f} MB")

## 3. Sanity checks

**Self-retrieval.** Search with the exact text of 200 random chunks. Each should find itself at rank 1; a few misses are fine when two chunks overlap a lot.
A low number would mean the index and the chunk table are out of sync.

In [ ]:
rng = np.random.default_rng(0)
picked = rng.choice(len(chunks), size=min(200, len(chunks)), replace=False)
_, top1 = index.search(embeddings[picked], 1)
print(f"self-retrieval at rank 1: {np.mean(top1[:, 0] == picked):.1%}")

**Eyeball test.** A few real questions, dense search only (no reranking yet, that is notebook 03).

In [ ]:
for query in ["What are the symptoms of type 2 diabetes?", "How is high blood pressure treated?", "What causes asthma attacks?"]:
    start = time.perf_counter()
    vector = encoder.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, ids = index.search(vector, 3)
    print(f"\n{query}   ({(time.perf_counter() - start) * 1000:.0f} ms)")
    for score, i in zip(scores[0], ids[0]):
        print(f"  {score:.3f}  {chunks.iloc[i]['source']} - {chunks.iloc[i]['title']}")

**Next:** `03_rag_pipeline.ipynb` loads this index and walks one question through the whole pipeline.